<p style="text-align:center; color:#525252; font-size:2.25rem; font-weight:bold; margin:0.35em 0;">wherewhen — CRS demos</p>

<p style="text-align:center; color:#525252; font-size:1.15rem; font-weight:bold; margin:0.35em 0;">China (GCJ-02 / BD-09) and Russia (SK-42) · wherewhen 0.2.6</p>

**Draft notebook** — companion to **wherewhen**, not an h3tools product notebook.

### Why a separate notebook?

| Live here (wherewhen) | Stay in h3tools |
|---|---|
| GCJ-02 / BD-09 / SK-42 conversions | Hex indexing, paths, hotspots |
| “Wrong pin on WGS-84 basemap” story | Place-based analytics once coords are WGS-84 |
| Ingest hygiene for China / Russia feeds | FIRMS, ADS-B, London OSM demos |

CRS is foundation work. After points are in **WGS-84**, hand them to **h3tools** (optional bridge at the end).

### What you will do

1. Load small demo CSVs (China landmarks + Baidu-style ingest + Russia landmarks)
2. Convert GCJ-02 / BD-09 → WGS-84 and measure offsets
3. Round-trip SK-42 ↔ WGS-84
4. *(Optional)* index corrected points with h3tools

### Data files (under `00 data/`)

| File | Role |
|---|---|
| `wherewhen_crs_demo_cn.csv` | Landmarks with WGS-84 + generated GCJ-02 / BD-09 columns |
| `wherewhen_crs_demo_bd09_ingest.csv` | BD-09-only ingest (lon, lat) — stand-in for Baidu Get Point |
| `wherewhen_crs_demo_ru.csv` | Russian landmarks with SK-42 columns via `pyproj` |

> **Note:** Demo GCJ/BD/SK-42 columns were produced with `wherewhen.crs` from approximate WGS-84 landmarks so the notebook is self-contained. For a larger *native* GCJ-02 POI corpus, see the Figshare tourist-mobility networks dataset (CC BY 4.0). For paired empirical Google↔OSM control points, see [gcj02-distortion-map `empirical-data.csv`](https://codeberg.org/leifgehrmann/gcj02-distortion-map) (MIT).

### Caveat

Chinese electronic-map regulations require GCJ-02 for approved map services; BD-09 is Baidu’s further obfuscation. These demos are for **interop / analyst hygiene**, not official surveying.


## **1. Paths and imports**

Set `TOOLKIT_PATH` to the parent of `wherewhen/` (and optionally `viztools/`, `h3-tools/`). Set `DATA_PATH` to the folder with the CRS CSVs.


In [ ]:
from pathlib import Path

# Parent of wherewhen/ (and optionally viztools/, h3-tools/)
TOOLKIT_PATH = Path("SET LOCAL PATH TO LIBRARY")
# Example:
TOOLKIT_PATH = Path("/Users/kennimus/Desktop/Working/grok")

# Folder containing the wherewhen_crs_demo_*.csv files
DATA_PATH = Path("SET LOCAL PATH TO DATA")
# Example (this workspace):
DATA_PATH = Path("/Users/kennimus/Desktop/Working/grok testing/00 data")

for label, path in [
    ("wherewhen", TOOLKIT_PATH / "wherewhen"),
    ("cn csv", DATA_PATH / "wherewhen_crs_demo_cn.csv"),
    ("bd csv", DATA_PATH / "wherewhen_crs_demo_bd09_ingest.csv"),
    ("ru csv", DATA_PATH / "wherewhen_crs_demo_ru.csv"),
]:
    mark = "✅" if path.exists() else "❌"
    print(f"{mark} {label}: {path}")


In [ ]:
import sys
from pathlib import Path

from viztools import *

# Editable install if needed (uncomment):
# import os
# os.environ.setdefault("PIP_EDITABLE_MODE", "compat")
# !{sys.executable} -m pip install -q -e "{TOOLKIT_PATH / 'wherewhen'}"
# !{sys.executable} -m pip install -q pyproj matplotlib pandas

import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point

from wherewhen.geometry import latlon_to_point, point_distance, point_to_latlon
from wherewhen.crs import (
    convert_crs,
    wgs84_to_cn_gcj02,
    cn_gcj02_to_wgs84,
    wgs84_to_cn_bd09,
    cn_bd09_to_wgs84,
    wgs84_to_ru_sk42,
    ru_sk42_to_wgs84,
)

print("wherewhen ready")


## **2. China — GCJ-02 offset (wrong pin vs corrected)**

Chinese map / POI feeds are often **GCJ-02**. Plotting those lon/lat on a **WGS-84** mental model (OSM, GPS, H3) shifts pins by hundreds of metres.

Workflow: treat `lat_gcj02` / `lon_gcj02` as the *ingest* CRS → `cn_gcj02_to_wgs84` → compare to known WGS-84.


In [ ]:
cn = pd.read_csv(DATA_PATH / "wherewhen_crs_demo_cn.csv")
cn.head()


In [ ]:
rows = []
for r in cn.itertuples(index=False):
    gcj_pt = latlon_to_point((r.lat_gcj02, r.lon_gcj02))
    wgs_true = latlon_to_point((r.lat_wgs84, r.lon_wgs84))
    wgs_from_gcj = cn_gcj02_to_wgs84(gcj_pt)
    rows.append({
        "name": r.name,
        "offset_raw_gcj_vs_wgs_m": point_distance(gcj_pt, wgs_true, units="m"),
        "roundtrip_err_m": point_distance(wgs_from_gcj, wgs_true, units="m"),
        "lat_corrected": wgs_from_gcj.y,
        "lon_corrected": wgs_from_gcj.x,
    })

summary = pd.DataFrame(rows)
summary


In [ ]:
# Visual: raw GCJ plotted as if WGS-84 (wrong) vs corrected WGS-84 (right)
fig, ax = plt.subplots(figsize=(7, 6))

for r in cn.itertuples(index=False):
    ax.scatter(r.lon_gcj02, r.lat_gcj02, c="#e6550d", s=10, zorder=3, label="_wrong")
    ax.scatter(r.lon_wgs84, r.lat_wgs84, c="#3182bd", s=10, zorder=3, label="_right")
    ax.plot(
        [r.lon_gcj02, r.lon_wgs84],
        [r.lat_gcj02, r.lat_wgs84],
        color="#969696",
        lw=0.8,
        zorder=2,
    )
    ax.annotate(r.name.split("_")[0], (r.lon_wgs84, r.lat_wgs84), fontsize=8, color="#525252")

plt.xlim(104.050,104.075)
plt.ylim(30.56,30.58)

# legend proxies
ax.scatter([], [], c="#e6550d", s=50, label="GCJ-02 plotted as WGS-84 (wrong)")
ax.scatter([], [], c="#3182bd", s=50, label="WGS-84 landmark (correct)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("China CRS demo — GCJ-02 offset vs WGS-84")
ax.legend(loc="best", fontsize=8)
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
format_plot(ax)
plt.show()


## **3. BD-09 ingest (Baidu-style)**

Baidu Maps / Get Point returns **BD-09** (often **longitude first** in the UI). Notebook pattern: load BD-09 → `cn_bd09_to_wgs84` → proceed in WGS-84.


In [ ]:
bd = pd.read_csv(DATA_PATH / "wherewhen_crs_demo_bd09_ingest.csv")
bd


In [ ]:
converted = []
for r in bd.itertuples(index=False):
    # Baidu-style: lon, lat
    bd_pt = Point(r.lon_bd09, r.lat_bd09)
    wgs = cn_bd09_to_wgs84(bd_pt)
    # Same via dispatcher
    wgs2 = convert_crs(bd_pt, "bd09", "wgs84")
    converted.append({
        "name": r.name,
        "lat_wgs84": wgs.y,
        "lon_wgs84": wgs.x,
        "dispatcher_match": point_distance(wgs, wgs2, units="m") < 1e-6,
    })

pd.DataFrame(converted)


## **4. Russia — SK-42 (Pulkovo 1942) round-trip**

`wherewhen.crs` maps geographic SK-42 (`EPSG:4284`) ↔ WGS-84 via **pyproj**.  
Native open SK-42 POI dumps are rare; this demo starts from WGS-84 landmarks, converts to SK-42, and round-trips.


In [ ]:
ru = pd.read_csv(DATA_PATH / "wherewhen_crs_demo_ru.csv")
ru


In [ ]:
rows = []
for r in ru.itertuples(index=False):
    wgs = latlon_to_point((r.lat_wgs84, r.lon_wgs84))
    sk = latlon_to_point((r.lat_sk42, r.lon_sk42))
    back = ru_sk42_to_wgs84(sk)
    forward = wgs84_to_ru_sk42(wgs)
    rows.append({
        "name": r.name,
        "csv_roundtrip_err_m": point_distance(back, wgs, units="m"),
        "live_roundtrip_err_m": point_distance(ru_sk42_to_wgs84(forward), wgs, units="m"),
        "sk42_lat": forward.y,
        "sk42_lon": forward.x,
    })

pd.DataFrame(rows)


## **5. Optional bridge — corrected WGS-84 → H3**

Only if `h3tools` is installed. Shows why CRS hygiene belongs **before** place analytics.


In [ ]:
try:
    from h3tools import point_to_h3, h3_to_point
    H3_OK = True
except ImportError:
    H3_OK = False
    print("❌ h3tools not importable — skip §5 or install the toolkit stack")

if H3_OK:
    res = 8
    bridge = []
    for r in cn.itertuples(index=False):
        wrong_cell = point_to_h3(latlon_to_point((r.lat_gcj02, r.lon_gcj02)), res)
        right_cell = point_to_h3(latlon_to_point((r.lat_wgs84, r.lon_wgs84)), res)
        bridge.append({
            "name": r.name,
            "h3_if_gcj_mistaken_as_wgs": wrong_cell,
            "h3_after_crs_fix": right_cell,
            "same_cell": wrong_cell == right_cell,
        })
    print(pd.DataFrame(bridge).to_string(index=False))
    n_diff = sum(not row["same_cell"] for row in bridge)
    print(f"ℹ️ At res={res}, {n_diff}/{len(bridge)} landmarks land in a different H3 cell if GCJ is treated as WGS-84.")


## **6. Takeaways**

1. **Import Chinese map coordinates as GCJ-02 or BD-09**, convert to WGS-84 before OSM / GPS / H3.
2. **SK-42** is geographic Pulkovo via pyproj in wherewhen — keep Gauss–Kruger *projected* metres as a separate step if you ever ingest those.
3. Keep this notebook under **wherewhen**; point h3tools users here when ingest spans China or former-USSR datums.
4. For larger native GCJ-02 POIs: Figshare tourist mobility networks; for empirical pairs: Codeberg `empirical-data.csv`.

---

*Draft — run cells after setting `TOOLKIT_PATH` and `DATA_PATH`. Sample CSVs are synthetic-from-landmarks for a self-contained demo.*
